# Classification de texte avec des RNNs

Dans ces travaux pratiques, nous allons utiliser des RNNs pour retrouver l'auteur d'un texte.

Le corpus utilisé est tiré d'œuvres littéraires classiques de littérature anglophone : 9 auteurs, 2 livres par auteurs avec un fichier par chapitre.

## Téléchargement des données depuis un répo git

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-9classical-author.git

## Téléchargement du modèle spacy pour l'anglais

In [ ]:
spacy_updated = !pip freeze | grep spacy==3
if not spacy_updated:
  !pip install 'spacy >= 3, < 4'
!python -m spacy download en_core_web_sm

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import collections
import collections.abc
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import torch
import torchmetrics
import tqdm.notebook
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Prétraitements

- Chaque phrase est un exemple
- Sont remplacés par leur type (avec la première lettre triplée pour créer de nouveaux « mots ») tout mot (ou groupe de mots) étant une entité nommée

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"))
nlp.enable_pipe("senter")


def replace_ners(tokens: collections.abc.Iterable[spacy.tokens.Token]
                 ) -> collections.abc.Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path,
             ) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  paths = list(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(zip(paths, nlp.pipe(contents,
                                                          n_process=-1)),
                                      total=len(paths)):

    author = path.parent.parent.name
    for sentence in doc.sents:
      authors.append(author)
      texts.append(" ".join(replace_ners(sentence)))

  return texts, authors


texts, authors = get_data(pathlib.Path("dataset-9classical-author"))

label_encoder = sklearn.preprocessing.LabelEncoder()
y = torch.tensor(label_encoder.fit_transform(authors))

print("Nombre de textes et forme de y :", len(texts), tuple(y.shape))

X_train_raw, X_test_raw, y_train, y_test = \
    sklearn.model_selection.train_test_split(texts, y, test_size=0.3)

In [ ]:
print(len(X_train_raw), tuple(y_train.shape))

## Préparation des données en séquences



### Préparation du dictionnaire et des séquences

À l'aide de la classe `Tokenizer` définie dans la cellule suivante :
- Constituez un dictionnaire sur le corpus `X_train_raw`
- Quelle est la taille du vocabulaire ?
- Quelle est la taille maximum, en nombre de mots, d'une phrase ?

In [ ]:
class Tokenizer:
  """Map each word of a corpus to an index, by decreasing frequency."""

  # Caractères retirés du texte (remplacés par des espaces) avant le découpage
  filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

  def __init__(self) -> None:
    self.table = str.maketrans(self.filters, " " * len(self.filters))
    self.word_index: dict[str, int] = {}

  def split(self, text: str) -> list[str]:
    return text.lower().translate(self.table).split()

  def fit_on_texts(self, texts: collections.abc.Iterable[str]) -> None:
    counts = collections.Counter(word
                                 for text in texts
                                 for word in self.split(text))
    # L'indice 0 est réservé au padding, les mots sont donc numérotés à partir
    # de 1
    self.word_index = {
        word: index
        for index, (word, _) in enumerate(counts.most_common(), start=1)}

  def texts_to_sequences(self,
                         texts: collections.abc.Iterable[str]
                         ) -> list[list[int]]:
    return [[self.word_index[word]
             for word in self.split(text)
             if word in self.word_index]
            for text in texts]

In [ ]:
# Votre code ici

#### Solution

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train_raw)

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(s.split()) for s in X_train_raw)
print(f"Taille du vocabulaire : {vocab_size}")
print(f"Taille de la plus grande séquence de mot : {max_length}")

### Création de la matrice de séquences d'indices

À l'aide de la classe `Tokenizer` et de la fonction `pad_sequences` définie dans la cellule suivante :
- Transformez `X_train_raw` et `X_test_raw` en séquences d'indices de mots dans un dictionnaire. (méthode ``texts_to_sequences()`` de l'objet ``Tokenizer`` instancié précédemment)
- Effectuez l'opération de **padding** sur les séquences afin qu'elles aient une taille raisonnable pour être traitées par un GRU bidirectionnel :
  - On utilisera `maxlen = 150` car au-delà les GRU commencent à ne plus être capable de transmettre correctement l'information.
- Stockez les séquences obtenues dans `X_train_pad` et `X_test_pad`

In [ ]:
def pad_sequences(sequences: collections.abc.Iterable[list[int]],
                  maxlen: int) -> torch.Tensor:
  """Truncate and pad each sequence at its beginning, to maxlen indices."""
  sequences = list(sequences)
  padded = torch.zeros(len(sequences), maxlen, dtype=torch.long)
  for i, sequence in enumerate(sequences):
    truncated = torch.tensor(sequence[-maxlen:], dtype=torch.long)
    padded[i, maxlen - len(truncated):] = truncated
  return padded

In [ ]:
# Votre code ici

#### Solution

In [ ]:
max_length = 150

X_train_tokens = tokenizer.texts_to_sequences(X_train_raw)
X_test_tokens = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_tokens, maxlen=max_length)
X_test_pad = pad_sequences(X_test_tokens, maxlen=max_length)

In [ ]:
print(f"Forme du corpus de documents : {tuple(X_train_pad.shape)}")
print(f"Premier exemple : {X_train_pad[0]}")

## Modélisation

Construisez un modèle avec pour caractéristiques :
  - Une couche d'embeddings de taille 300
  - Une couche de [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) bidirectionnels (option `bidirectional=True`) :
    - de taille `64`
    - une initialisation **des** matrices de poids orthogonales
    - un dropout à `0.2` appliqué aux embeddings, en entrée de la couche récurrente
  - Une couche de réseau de neurones dense qui prend en entrée les derniers états cachés des deux directions du [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)
  - Une fonction de perte basée sur l'**entropie croisée**
  - L'optimiseur `adam`
  - l'`accuracy` comme métrique d'évaluation

In [ ]:
class TextClassifier(lightning.LightningModule):
  """Lightning wrapper: predict the author of a padded sequence of words."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               sequence_len: int = max_length,
               num_classes: int = len(label_encoder.classes_)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, sequence_len, dtype=torch.long)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    return self.model(sequences)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    sequences, authors = batch
    logits = self(sequences)
    loss = nn.functional.cross_entropy(logits, authors)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, authors)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: torch.Tensor | tuple[torch.Tensor, ...],
                   batch_index: int) -> torch.Tensor:
    sequences = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(sequences).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)

In [ ]:
# Votre code ici

### Solution

In [ ]:
EMBEDDING_DIM = 300


def orthogonal_(rnn: nn.RNNBase) -> nn.RNNBase:
  """Initialize the weight matrices of a recurrent layer orthogonally."""
  for name, parameter in rnn.named_parameters():
    if name.startswith("weight"):
      nn.init.orthogonal_(parameter)
  return rnn


class TextGRU(nn.Module):
  """Embed the words, summarize them with a GRU, score each author."""

  def __init__(self,
               embedding: nn.Embedding,
               hidden_size: int = 64,
               num_layers: int = 1,
               bidirectional: bool = True,
               dropout: float = 0.2,
               num_classes: int = 9) -> None:
    super().__init__()
    self.embedding = embedding
    # Dropout appliqué aux embeddings, en entrée de la couche récurrente
    self.dropout = nn.Dropout(dropout)
    self.gru = orthogonal_(
        nn.GRU(embedding.embedding_dim,
               hidden_size,
               num_layers=num_layers,
               bidirectional=bidirectional,
               dropout=dropout if num_layers > 1 else 0.,
               batch_first=True))
    self.head = nn.Linear(hidden_size * (2 if bidirectional else 1),
                          num_classes)

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    _, hidden = self.gru(embedded)
    if self.gru.bidirectional:
      # Concaténation des derniers états cachés des deux directions
      summary = torch.cat([hidden[-2], hidden[-1]], dim=-1)
    else:
      summary = hidden[-1]
    return self.head(summary)


model = TextClassifier(TextGRU(nn.Embedding(vocab_size, EMBEDDING_DIM)))
print(ModelSummary(model, max_depth=-1))

## Apprentissage

Apprenez votre modèle sur `X_train_pad` :
- avec des batch de taille `1024`
- pendant `10` itérations
- en utilisant 30% de la base d'apprentissage pour validation


In [ ]:
# Votre code ici

### Solution

In [ ]:
batch_size = 1024

train_dataset, val_dataset = random_split(
    TensorDataset(X_train_pad, y_train), [0.7, 0.3])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Évaluation

Évaluez les performances de votre modèle sur la base de test.

In [ ]:
# Votre code ici

### Solution

In [ ]:
test_loader = DataLoader(TensorDataset(X_test_pad, y_test),
                         batch_size=batch_size)
test_accuracy = trainer.test(model, test_loader)[0]["test_accuracy"]
print(f"Accuracy sur la base de test : {test_accuracy:.3f}")

## Premières conclusions

*Au vu des résultats, que peut-on dire de la qualité de cet apprentissage ? Justifiez.*

Votre réponse ici

### Solution

On observe un phénomène de surapprentissage. train $\approx$ 75% alors que validation et test $\approx$ 50%


## Utilisation d'embeddings de mots pré-appris

Afin d'obtenir de meilleurs résultats, vous allez utiliser les embeddings pré-appris GloVe, plutôt que d'apprendre des embeddings spécifiques comme précédemment.

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
def glove_path(embedding_dim: int) -> pathlib.Path:
  if embedding_dim in {50, 100, 200, 300}:
    return pathlib.Path("glove.6B.{}d.txt".format(embedding_dim))
  else:
    raise ValueError("embedding_dim must be in {50, 100, 200, 300}")

## Réutilisation des Word Embeddings GloVe

La couche [`nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html) de PyTorch peut être initialisée avec une matrice de poids où la ligne i correspond à l'embedding du mot i.

Après un rapide coup d'oeil aux fichiers `glove.6B.300d.txt` :
- Créez un layer `embedding_layer` de type [`nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html) :
  - Initialisez-le avec les embeddings GloVe.
  - Initialisez les mots de notre vocabulaire qui ne seraient pas dans GloVe avec le vecteur nul
  - Utilisez le flag approprié pour empêcher le changement de ces embeddings pendant l'apprentissage

In [ ]:
# Votre code ici

### Solution

In [ ]:

EMBEDDING_DIM = 300
embedding_matrix = numpy.zeros((vocab_size, EMBEDDING_DIM), dtype="float32")
found = 0
with glove_path(EMBEDDING_DIM).open() as fh:
  for line in fh:
    values = line.split(" ")
    word = values[0]
    if word in tokenizer.word_index:
      found += 1
      coeffs = numpy.array(values[1:], dtype="float32")
      embedding_matrix[tokenizer.word_index[word]] = coeffs

print(f"Utilisation de {found} embeddings pré-entraînés sur {vocab_size} mots "
      "dans le vocabulaire")

embedding_layer = nn.Embedding.from_pretrained(
    torch.from_numpy(embedding_matrix), freeze=True)

## Modélisation, apprentissage et évaluation

- Créez un modèle identique à celui de la partie précédente, à ceci près que la couche d'embeddings est celle que l'on vient d'instancier à partir de GloVe
- Entraînez ce modèle avec les mêmes paramètres que dans la partie précédente
- Évaluez-le sur les données de test

NB : pour obtenir de meilleurs résultats qu'avec des SVMs, un modèle est proposé en solution.

In [ ]:
# Votre code ici

### Solution

In [ ]:
model = TextClassifier(TextGRU(embedding_layer, dropout=0.2))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=20,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="glove_bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.test(model, test_loader)

In [ ]:
EMBEDDING_DIM = 300

model = TextClassifier(
    TextGRU(embedding_layer, bidirectional=False, dropout=0.))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="glove_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Solution meilleure que des SVM

In [ ]:
model = TextClassifier(
    TextGRU(embedding_layer, num_layers=2, dropout=0.5))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=30,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="deep_bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.test(model, test_loader)

## Affichage de la matrice de confusion en Test

*Affichez la matrice de confusion (l'utilisation d'une heatmap est recommandée).*

In [ ]:
# Votre code ici

### Solution

In [ ]:
y_pred_softmax = torch.cat(trainer.predict(model, test_loader))
y_pred = y_pred_softmax.argmax(dim=1)

In [ ]:
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_pred)
seaborn.heatmap(conf_mat,
                annot=True,
                fmt="d",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cmap="rocket_r")
plt.show()